In [2]:
# ==============================================================================
# CELL: READ DATA & REPRODUCE ABSORBER STATISTICS TABLE (TNG50-4)
# ==============================================================================
import pandas as pd
import numpy as np

In [9]:
# --- 1. CONFIGURAÇÃO DO ARQUIVO ---
# Coloque o nome exato do arquivo gerado pelo run_parallel_grid_TNG.py
# Como usamos SKEWERS_PER_AXIS = 250_000, o total será 750_000.
file_path = '/home/vroberto/FakeSpectra_DLAs/notebooks_TNG/TNG50-4_z3_Grid_750000Sk.pkl' 

print("Loading the file...")
df = pd.read_pickle(file_path)

# --- 2. INSPEÇÃO DA ESTRUTURA ---
print("\n" + "="*50)
print(" ESTRUTURA DO DATAFRAME (Primeiras Linhas)")
print("="*50)
display(df.head())

print("\n" + "="*50)
print(" INFORMAÇÕES DAS COLUNAS")
print("="*50)
df.info()

Loading the file...

 ESTRUTURA DO DATAFRAME (Primeiras Linhas)


,skewer_idx,axis,pos_x_code,pos_y_code,pos_z_code,chunk_id,log_n_hi,total_col_den,flux,n_hi_col,temp_k,wave,vel_axis,chi_axis
3000,0,1,0.0,35.0,35.0,0,15.575921,3.766353e+15,"[0.15132178, 0.0373169, 0.02889727, 0.09854509...","[4854045000000.0, 6229671300000.0, 20644211000...","[9267.527, 10377.879, 11712.439, 12005.046, 12...","[1215.67, 1215.7106, 1215.7511, 1215.7917, 121...","[0.0, 10.003904, 20.007809, 30.011711, 40.0156...","[0.0, 88.383835, 176.76767, 265.15152, 353.535..."
3001,1,1,0.0,105.0,35.0,0,15.639004,4.355163e+15,"[0.14122112, 0.036254067, 0.03375259, 0.129336...","[4930268000000.0, 12827039000000.0, 1941767700...","[8715.249, 10363.107, 11200.733, 11592.768, 11...","[1215.67, 1215.7106, 1215.7511, 1215.7917, 121...","[0.0, 10.003904, 20.007809, 30.011711, 40.0156...","[0.0, 88.383835, 176.76767, 265.15152, 353.535..."
3002,2,1,0.0,175.0,35.0,0,15.795738,6.247954e+15,"[0.14160289, 0.027057705, 0.017441407, 0.07165...","[4934172000000.0, 10622341000000.0, 1516834600...","[8630.731, 9881.831, 11175.008, 11547.456, 118...","[1215.67, 1215.7106, 1215.7511, 1215.7917, 121...","[0.0, 10.003904, 20.007809, 30.011711, 40.0156...","[0.0, 88.383835, 176.76767, 265.15152, 353.535..."
3003,3,1,0.0,245.0,35.0,0,15.870550,7.422495e+15,"[0.22910145, 0.067090936, 0.047569036, 0.13369...","[4998785700000.0, 6404398600000.0, 12420861000...","[8302.465, 9442.613, 10525.248, 11281.648, 118...","[1215.67, 1215.7106, 1215.7511, 1215.7917, 121...","[0.0, 10.003904, 20.007809, 30.011711, 40.0156...","[0.0, 88.383835, 176.76767, 265.15152, 353.535..."
3004,4,1,0.0,315.0,35.0,0,15.797790,6.277543e+15,"[0.31033742, 0.15265302, 0.13338861, 0.2422988...","[3656354400000.0, 6365064000000.0, 71353090000...","[8217.949, 8966.1875, 10006.645, 10795.175, 11...","[1215.67, 1215.7106, 1215.7511, 1215.7917, 121...","[0.0, 10.003904, 20.007809, 30.011711, 40.0156...","[0.0, 88.383835, 176.76767, 265.15152, 353.535..."



 INFORMAÇÕES DAS COLUNAS
<class 'pandas.core.frame.DataFrame'>
Index: 750000 entries, 3000 to 749999
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   skewer_idx     750000 non-null  int64  
 1   axis           750000 non-null  int64  
 2   pos_x_code     750000 non-null  float64
 3   pos_y_code     750000 non-null  float64
 4   pos_z_code     750000 non-null  float64
 5   chunk_id       750000 non-null  int64  
 6   log_n_hi       750000 non-null  float64
 7   total_col_den  750000 non-null  float64
 8   flux           750000 non-null  object 
 9   n_hi_col       750000 non-null  object 
 10  temp_k         750000 non-null  object 
 11  wave           750000 non-null  object 
 12  vel_axis       750000 non-null  object 
 13  chi_axis       750000 non-null  object 
dtypes: float64(5), int64(3), object(6)
memory usage: 85.8+ MB


In [10]:
# --- 3. CÁLCULO DA DENSIDADE TOTAL ---
# NOTA SOBRE UNIDADES: Como injetamos o UnitSystem no script de extração, 
# a coluna 'total_col_den' já foi convertida pelo fake_spectra para átomos/cm^2 (físico).
# Portanto, a binagem ocorrerá perfeitamente sem precisar de fatores multiplicativos aqui!
df['total_N_HI_skewer'] = df['total_col_den']

# --- 4. CLASSIFICAÇÃO DOS SKEWERS (BINAGEM) ---
# Definindo os limites exatos fornecidos pela imagem (em escala linear)
# N_HI em [atoms cm^-2]
bins = [0, 1.6e17, 1e19, 2e20, 1e21, np.inf]
labels = ['Lyman-α forest', 'LLS', 'Sub-DLA', 'Small DLA', 'Large DLA']

# right=False garante que o intervalo seja [min, max), ou seja, inclui o mínimo e exclui o máximo, 
# assim como N(HI)_min <= N < N(HI)_max
df['category'] = pd.cut(df['total_N_HI_skewer'], bins=bins, labels=labels, right=False)

# --- 5. CÁLCULO DAS PORCENTAGENS ---
total_skewers = len(df)
# Conta quantos skewers caíram em cada categoria preservando a ordem dos labels
category_counts = df['category'].value_counts(sort=False) 
category_percentages = (category_counts / total_skewers) * 100

# Avaliando se a malha resolveu o problema dos zeros:
how_many_zeros = np.sum(df['total_N_HI_skewer'] == 0)
print(f"\nQuantidade de skewers com densidade EXATAMENTE zero: {how_many_zeros} (de {total_skewers})")


Quantidade de skewers com densidade EXATAMENTE zero: 0 (de 750000)


In [11]:
# --- 6. MONTAGEM DA TABELA FINAL ---
summary_table = pd.DataFrame({
    'Absorber category': labels,
    'N(HI)_min [cm^-2]': ['0', '1.6 x 10^17', '1 x 10^19', '2 x 10^20', '1 x 10^21'],
    'N(HI)_max [cm^-2]': ['1.6 x 10^17', '1 x 10^19', '2 x 10^20', '1 x 10^21', '∞'],
    'Counts (Simulated)': category_counts.values,
    f'% of spectra in box (Simulated)': category_percentages.values
})

# Formatação estética para % (1 casa decimal)
summary_table[f'% of spectra in box (Simulated)'] = summary_table[f'% of spectra in box (Simulated)'].map('{:.1f}'.format)

print("\n" + "="*80)
print(" TABELA DE ABSORVEDORES REPRODUZIDA (TNG50-4, z=3)")
print("="*80)
display(summary_table)

# Opcional: Verificação rápida de sanidade (se a soma fecha em 100%)
soma_pct = category_percentages.sum()
print(f"\nSoma total das frações: {soma_pct:.1f}%")


 TABELA DE ABSORVEDORES REPRODUZIDA (TNG50-4, z=3)


,Absorber category,N(HI)_min [cm^-2],N(HI)_max [cm^-2],Counts (Simulated),% of spectra in box (Simulated)
0,Lyman-α forest,0,1.6 x 10^17,683848,91.2
1,LLS,1.6 x 10^17,1 x 10^19,48843,6.5
2,Sub-DLA,1 x 10^19,2 x 10^20,12626,1.7
3,Small DLA,2 x 10^20,1 x 10^21,3725,0.5
4,Large DLA,1 x 10^21,∞,958,0.1



Soma total das frações: 100.0%
